# Modelica nonlinear FMU inside DPsim EMT three-phase MNA

Three phase-shifted instances of Modelica Standard Library 4.0.0's `ShowSaturatingInductor` provide the **native Modelica reference**. Each phase contains a nonlinear saturating inductor in parallel with a linear inductor. The FMU replaces only the prescribed source with scalar voltage/current ports; component equations, parameters and zero-current initialization are preserved.

DPsim's generic `NonlinearSSN` imports six states from the FMI 3.0 FMU, applies implicit trapezoidal integration and solves its current relation in the MNA topology. A prescribed-voltage helper imposes the original 1.25 V, 1 rad/s source waveforms. No inductor equations are implemented in C++. This is an exporter-portability and nonlinear-branch check; the source frequency intentionally follows the original MSL example, rather than 50 Hz. Nonlinear phase currents can have a zero-sequence component.

The native reference uses OpenModelica's fixed-step RK4 at 100 microseconds. Coarse DPsim timesteps demonstrate trapezoidal convergence; the fine runs reach the six-decimal CSV rounding limit. All generated results stay under the SSD development directory.

In [ ]:
from pathlib import Path
import subprocess, os, sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO = Path("/Volumes/SSDMatthias/development/dpsim")
ROOT = REPO / "outputs/fmi-nonlinear/modelica"
sys.path.insert(0, str(REPO / "dpsim-fmi/tests"))
from compare_modelica import compare

plt.rcParams.update({"figure.dpi": 120, "axes.grid": True, "grid.alpha": 0.25})

In [ ]:
# Reuses saved native Modelica reference; reruns all imported FMU cases.
env = os.environ.copy()
env["TMPDIR"] = str(REPO / "outputs/fmi-nonlinear/tmp")
exe = REPO / "build-macos-check/dpsim-fmi/FMI_EMT_Ph3_Prescribed"
for name, h in [
    ("dpsim-10ms", 0.01),
    ("dpsim-5ms", 0.005),
    ("dpsim-200us", 0.0002),
    ("dpsim-100us", 0.0001),
]:
    folder = ROOT / name
    folder.mkdir(exist_ok=True)
    with (folder / "run.log").open("w") as log:
        subprocess.run(
            [str(exe), str(ROOT / "Ported.fmu"), str(ROOT / "benchmark.json"), str(h)],
            cwd=folder,
            env=env,
            stdout=log,
            stderr=subprocess.STDOUT,
            check=True,
        )
metrics = compare(ROOT)
pd.DataFrame(metrics["runs"]).T

In [ ]:
reference = pd.read_csv(ROOT / "NonlinearThreePhase.Reference_res.csv").drop_duplicates(
    "time", keep="last"
)


def read(name):
    data = pd.read_csv(ROOT / name / "logs/fmi_prescribed.csv", skipinitialspace=True)
    data.columns = data.columns.str.strip()
    return data


fine = read("dpsim-200us")
fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
for ax, p in zip(axes, "abc"):
    ax.plot(reference.time, reference["i_" + p], label="native MSL reference", lw=2)
    ax.plot(
        fine.time, fine["i_" + p], "--", label="DPsim FMI, trapezoidal 200 µs", lw=1
    )
    ax.set_ylabel("Phase " + p + " current [A]")
axes[0].legend()
axes[-1].set_xlabel("Time [s]")
fig.tight_layout()
fig.savefig(ROOT / "waveforms.png", bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
for name, label in [("dpsim-10ms", "10 ms"), ("dpsim-5ms", "5 ms")]:
    data = read(name)
    error = data.i_a - np.interp(data.time, reference.time, reference.i_a)
    axes[0].plot(data.time, error * 1e6, label=label)
axes[0].set_ylabel("Phase-A error [µA]")
axes[0].legend()
error = fine.i_a - np.interp(fine.time, reference.time, reference.i_a)
axes[1].plot(fine.time, error * 1e6, label="200 µs")
axes[1].set_ylabel("Fine-run error [µA]")
axes[1].set_xlabel("Time [s]")
axes[1].legend()
fig.tight_layout()
fig.savefig(ROOT / "errors.png", bbox_inches="tight")
plt.show()
print("RMS error reduction on halving h:", metrics["rms_refinement_ratio"])
assert metrics["passed"]

The independent native simulation and imported FMU agree for this smooth nonlinear MSL circuit. This does not establish support for all Modelica models. Hybrid events, FMI clocks, arrays, continuous-state reinitialization and internal scheduled events are outside the currently supported subset. See `dpsim-fmi/NONLINEAR.md` and the Modelica example README for provenance and reproduction.